In [ ]:
!git clone https://github.com/aria-yang/youtube-thumbnail-performance-predictor.git
%cd /content/youtube-thumbnail-performance-predictor
!git checkout regression

Cloning into 'youtube-thumbnail-performance-predictor'...
remote: Enumerating objects: 14410, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (31/31), done.


Install Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q numpy pandas pillow tqdm scikit-learn loguru typer python-dotenv
!pip install -q easyocr facenet-pytorch deepface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.0 MB/s eta 0:00:00


Run full merged pipeline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, shutil
from pathlib import Path

# Define the source directory on Google Drive
artifact_root = Path('/content/drive/MyDrive/ECE324/youtube-thumbnail-performance-predictor-artifacts')

# Define the destination directory in the cloned repository
local_processed_data_path = Path('/content/youtube-thumbnail-performance-predictor/data/processed')
local_processed_data_path.mkdir(parents=True, exist_ok=True)

# List of files to copy from Google Drive to local processed data folder
files_to_copy_from_drive = [
    'merged_cnn_cache_resnet50.csv',
    'merged_cnn_embeddings_resnet50.npy',
    'merged_labeled_data.csv',
    'merged_ocr_features.csv',
    'merged_text_embeddings.npy',
    'merged_face_cache.csv',
    'merged_face_embeddings.npy',
]

for filename in files_to_copy_from_drive:
    src_path = artifact_root / filename
    dst_path = local_processed_data_path / filename
    if src_path.exists():
        shutil.copy2(src_path, dst_path)
        print(f'Copied {filename} from Google Drive to local data/processed')
    else:
        print(f'Missing in Google Drive artifacts: {filename}')

Copied merged_cnn_cache_resnet50.csv from Google Drive to local data/processed
Copied merged_cnn_embeddings_resnet50.npy from Google Drive to local data/processed
Copied merged_labeled_data.csv from Google Drive to local data/processed
Copied merged_ocr_features.csv from Google Drive to local data/processed
Copied merged_text_embeddings.npy from Google Drive to local data/processed
Copied merged_face_cache.csv from Google Drive to local data/processed
Copied merged_face_embeddings.npy from Google Drive to local data/processed


In [ ]:
!git pull
!PYTHONPATH=. python training/train_fusion_regression.py \
  --device auto \
  --split_name random \
  --target_transform log1p \
  --loss smoothl1 \
  --num_epochs 50 \
  --batch_size 128 \
  --artifact_root /content/drive/MyDrive/ECE324/youtube-thumbnail-performance-predictor-artifacts

Already up to date.
2026-03-29 03:02:10.841 | INFO     | thumbnail_performance.config:<module>:11 - PROJ_ROOT path is: /content/youtube-thumbnail-performance-predictor
Epoch 001 | Train Loss: 0.1800 | Val Loss: 0.1248 | Val MAE: 0.3521 | Val RMSE: 0.5480 | Val R2: 0.0658 | Val Spearman: 0.6626
  ✓ Val loss improved inf → 0.1248. Saving model.
Epoch 002 | Train Loss: 0.1124 | Val Loss: 0.0865 | Val MAE: 0.2869 | Val RMSE: 0.4373 | Val R2: 0.4051 | Val Spearman: 0.7219
  ✓ Val loss improved 0.1248 → 0.0865. Saving model.
Epoch 003 | Train Loss: 0.0835 | Val Loss: 0.0744 | Val MAE: 0.2598 | Val RMSE: 0.4090 | Val R2: 0.4798 | Val Spearman: 0.7596
  ✓ Val loss improved 0.0865 → 0.0744. Saving model.
Epoch 004 | Train Loss: 0.0666 | Val Loss: 0.0719 | Val MAE: 0.2663 | Val RMSE: 0.3993 | Val R2: 0.5041 | Val Spearman: 0.7201
  ✓ Val loss improved 0.0744 → 0.0719. Saving model.
Epoch 005 | Train Loss: 0.0547 | Val Loss: 0.0642 | Val MAE: 0.2450 | Val RMSE: 0.3724 | Val R2: 0.5686 | Val Spear

## Tuning

In [ ]:
!git pull
!PYTHONPATH=. python training/tune_fusion_regression.py \
  --device auto \
  --split_names random \
  --target_modes log1p,log1p_zscore \
  --losses smoothl1,mse \
  --batch_sizes 64,128 \
  --lrs 0.0005,0.001 \
  --dropouts 0.3,0.4 \
  --hidden_dims 512x256,1024x512 \
  --seeds 42,43 \
  --num_epochs 20 \
  --metric_to_rank val_spearman \
  --artifact_root /content/drive/MyDrive/ECE324/youtube-thumbnail-performance-predictor-artifacts

Streaming output truncated to the last 5000 lines.
  · No improvement for 1/7 epochs.
Epoch 013 | Train Loss: 0.0191 | Val Loss: 0.0565 | Val MAE: 0.2175 | Val RMSE: 0.3485 | Val R2: 0.6223 | Val Spearman: 0.7971
  · No improvement for 2/7 epochs.
Epoch 014 | Train Loss: 0.0179 | Val Loss: 0.0560 | Val MAE: 0.2192 | Val RMSE: 0.3483 | Val R2: 0.6227 | Val Spearman: 0.7951
  · No improvement for 3/7 epochs.
Epoch 015 | Train Loss: 0.0163 | Val Loss: 0.0553 | Val MAE: 0.2207 | Val RMSE: 0.3438 | Val R2: 0.6323 | Val Spearman: 0.7953
  ✓ Val loss improved 0.0554 → 0.0553. Saving model.
Epoch 016 | Train Loss: 0.0157 | Val Loss: 0.0548 | Val MAE: 0.2226 | Val RMSE: 0.3418 | Val R2: 0.6365 | Val Spearman: 0.7873
  ✓ Val loss improved 0.0553 → 0.0548. Saving model.
Epoch 017 | Train Loss: 0.0163 | Val Loss: 0.0548 | Val MAE: 0.2152 | Val RMSE: 0.3430 | Val R2: 0.6339 | Val Spearman: 0.8060
  · No improvement for 1/7 epochs.
Epoch 018 | Train Loss: 0.0157 | Val Loss: 0.0563 | Val MAE: 0.2217 

## Final Training

In [ ]:
!PYTHONPATH=. python training/train_fusion_regression.py \
  --device auto \
  --split_name random \
  --target_transform log1p \
  --loss mse \
  --batch_size 64 \
  --lr 0.001 \
  --num_epochs 60 \
  --seed 42 \
  --artifact_root /content/drive/MyDrive/ECE324/youtube-thumbnail-performance-predictor-artifacts \
  --checkpoint_path /content/drive/MyDrive/ECE324/final-regression/fusion_mlp_regression_final_seed42.pt \
  --metrics_path /content/drive/MyDrive/ECE324/final-regression/fusion_mlp_regression_final_seed42_metrics.json

2026-03-29 03:20:20.387 | INFO     | thumbnail_performance.config:<module>:11 - PROJ_ROOT path is: /content/youtube-thumbnail-performance-predictor
Epoch 001 | Train Loss: 0.3782 | Val Loss: 0.2556 | Val MAE: 0.3405 | Val RMSE: 0.5056 | Val R2: 0.2048 | Val Spearman: 0.6396
  ✓ Val loss improved inf → 0.2556. Saving model.
Epoch 002 | Train Loss: 0.2240 | Val Loss: 0.1729 | Val MAE: 0.2832 | Val RMSE: 0.4159 | Val R2: 0.4621 | Val Spearman: 0.6992
  ✓ Val loss improved 0.2556 → 0.1729. Saving model.
Epoch 003 | Train Loss: 0.1498 | Val Loss: 0.1686 | Val MAE: 0.2654 | Val RMSE: 0.4106 | Val R2: 0.4755 | Val Spearman: 0.7687
  ✓ Val loss improved 0.1729 → 0.1686. Saving model.
Epoch 004 | Train Loss: 0.1112 | Val Loss: 0.1351 | Val MAE: 0.2574 | Val RMSE: 0.3675 | Val R2: 0.5798 | Val Spearman: 0.7704
  ✓ Val loss improved 0.1686 → 0.1351. Saving model.
Epoch 005 | Train Loss: 0.0891 | Val Loss: 0.1279 | Val MAE: 0.2265 | Val RMSE: 0.3577 | Val R2: 0.6020 | Val Spearman: 0.7839
  ✓ Val 

#Ablation and SHAP

In [ ]:
print("Skipping the old plotting patch cell. The ablation script should be run directly with the current repo version.")

Patched /content/youtube-thumbnail-performance-predictor/training/ablation_study_regression.py


In [ ]:
!PYTHONPATH=. python training/ablation_study_regression.py \
  --device auto \
  --split_name random \
  --target_transform log1p \
  --loss mse \
  --batch_size 64 \
  --num_epochs 60 \
  --lr 0.001 \
  --hidden1 512 \
  --hidden2 256 \
  --dropout 0.4 \
  --seeds 42,43,44,45,46 \
  --ranking_metric val_spearman \
  --artifact_root /content/drive/MyDrive/ECE324/youtube-thumbnail-performance-predictor-artifacts \
  --output_dir /content/drive/MyDrive/ECE324/final-regression/ablation

Your local changes to the following files would be overwritten by merge:
  data/processed/merged_cnn_cache_resnet50.csv2026-03-29 02:04:10.180 | INFO     | thumbnail_performance.config:<module>:11 - PROJ_ROOT path is: /content/youtube-thumbnail-performance-predictor
2026-03-29 02:04:10.504 | INFO     | __main__:run_ablation_experiment:100 - Starting regression ablation study over 5 seeds on CUDA using split 'random'.
2026-03-29 02:04:10.505 | INFO     | __main__:run_ablation_experiment:106 - Evaluating configuration: CNN-only
Epoch 001 | Train Loss: 0.4396 | Val Loss: 0.3363 | Val MAE: 0.4169 | Val RMSE: 0.5800 | Val R2: -0.0463 | Val Spearman: 0.3635
  ✓ Val loss improved inf → 0.3363. Saving model.
Epoch 002 | Train Loss: 0.2681 | Val Loss: 0.2757 | Val MAE: 0.3888 | Val RMSE: 0.5250 | Val R2: 0.1425 | Val Spearman: 0.3769
  ✓ Val loss improved 0.3363 → 0.2757. Saving model.
Epoch 003 | Train Loss: 0.1996 | Val Loss: 0.2797 | Val MAE: 0.3814 | Val RMSE: 0.5289 | Val R2: 0.1298 | Val 

In [ ]:
!git pull
!PYTHONPATH=. python training/run_shap_regression.py \
  --device auto \
  --split_name random \
  --target_transform log1p \
  --loss mse \
  --batch_size 64 \
  --num_epochs 60 \
  --lr 0.001 \
  --hidden1 512 \
  --hidden2 256 \
  --dropout 0.4 \
  --seed 42 \
  --checkpoint_path /content/drive/MyDrive/ECE324/final-regression/fusion_mlp_regression_final_seed42.pt \
  --artifact_root /content/drive/MyDrive/ECE324/youtube-thumbnail-performance-predictor-artifacts \
  --output_dir /content/drive/MyDrive/ECE324/final-regression/shap_seed42

Your local changes to the following files would be overwritten by merge:
  data/processed/merged_cnn_cache_resnet50.csv2026-03-29 02:08:09.773 | INFO     | thumbnail_performance.config:<module>:11 - PROJ_ROOT path is: /content/youtube-thumbnail-performance-predictor
Loaded checkpoint from /content/drive/MyDrive/ECE324/final-regression/fusion_mlp_regression_final_seed42.pt
Saved global SHAP plot to /content/drive/MyDrive/ECE324/final-regression/shap_seed42/shap_regression_global_importance.png
Saved full feature ranking to /content/drive/MyDrive/ECE324/final-regression/shap_seed42/shap_regression_feature_importance.csv
Saved top 10 features to /content/drive/MyDrive/ECE324/final-regression/shap_seed42/shap_regression_top10_features.csv
Saved interpretation notes to /content/drive/MyDrive/ECE324/final-regression/shap_seed42/shap_regression_notes.md
Saved run metadata to /content/drive/MyDrive/ECE324/final-regression/shap_seed42/shap_regression_run_metadata.json

Top 10 features:
 rank   

In [ ]:
!PYTHONPATH=. python training/eval_crosssplit_regression.py \
  --device auto \
  --checkpoint_path /content/drive/MyDrive/ECE324/final-regression/fusion_mlp_regression_final_seed42.pt \
  --artifact_root /content/drive/MyDrive/ECE324/youtube-thumbnail-performance-predictor-artifacts \
  --output_dir /content/drive/MyDrive/ECE324/final-regression/cross_split_seed42

# GitHub and Google Drive

In [ ]:
print("Skipping GitHub commit/push in Colab. Export artifacts to Google Drive instead.")

In [ ]:
print("Artifacts should be copied to Drive, not pushed from Colab.")

GitHub token: ··········
Everything up-to-date


In [ ]:
import shutil
from pathlib import Path

artifact_root = Path("/content/drive/MyDrive/ECE324/youtube-thumbnail-performance-predictor-artifacts")
artifact_root.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    "/content/youtube-thumbnail-performance-predictor/data/processed/merged_cnn_embeddings_resnet50.npy",
    "/content/youtube-thumbnail-performance-predictor/data/processed/merged_labeled_data.csv",
    "/content/youtube-thumbnail-performance-predictor/data/processed/merged_ocr_features.csv",
    "/content/youtube-thumbnail-performance-predictor/data/processed/merged_text_embeddings.npy",
    "/content/youtube-thumbnail-performance-predictor/data/processed/merged_face_embeddings.npy",
    "/content/youtube-thumbnail-performance-predictor/data/processed/fusion_mlp_regression_metrics.json",
    "/content/youtube-thumbnail-performance-predictor/data/processed/fusion_regression_tuning_all_results.csv",
    "/content/youtube-thumbnail-performance-predictor/data/processed/fusion_regression_tuning_summary.csv",
    "/content/youtube-thumbnail-performance-predictor/data/processed/fusion_regression_tuning_summary.json",
    "/content/youtube-thumbnail-performance-predictor/models/fusion_mlp_regression.pt",
    "/content/drive/MyDrive/ECE324/final-regression/fusion_mlp_regression_final_seed42.pt",
    "/content/drive/MyDrive/ECE324/final-regression/fusion_mlp_regression_final_seed42_metrics.json",
    "/content/drive/MyDrive/ECE324/final-regression/ablation/ablation_regression_all_runs.csv",
    "/content/drive/MyDrive/ECE324/final-regression/ablation/ablation_regression_summary.csv",
    "/content/drive/MyDrive/ECE324/final-regression/ablation/ablation_regression_val_spearman.png",
    "/content/drive/MyDrive/ECE324/final-regression/shap_seed42/shap_regression_global_importance.png",
    "/content/drive/MyDrive/ECE324/final-regression/shap_seed42/shap_regression_feature_importance.csv",
    "/content/drive/MyDrive/ECE324/final-regression/shap_seed42/shap_regression_top10_features.csv",
    "/content/drive/MyDrive/ECE324/final-regression/shap_seed42/shap_regression_notes.md",
    "/content/drive/MyDrive/ECE324/final-regression/shap_seed42/shap_regression_run_metadata.json",
    "/content/drive/MyDrive/ECE324/final-regression/cross_split_seed42/cross_split_regression_generalization.csv",
    "/content/drive/MyDrive/ECE324/final-regression/cross_split_seed42/cross_split_regression_generalization.json",
    "/content/drive/MyDrive/ECE324/final-regression/cross_split_seed42/cross_split_regression_spearman.png",
]

for src in files_to_copy:
    src_path = Path(src)
    if src_path.exists():
        dst_path = artifact_root / src_path.name
        shutil.copy2(src_path, dst_path)
        print(f"Copied {src_path.name} -> {dst_path}")
    else:
        print(f"Missing locally: {src_path}")

Copied merged_cnn_embeddings_resnet50.npy -> /content/drive/MyDrive/ECE324/youtube-thumbnail-performance-predictor-artifacts/merged_cnn_embeddings_resnet50.npy
Copied merged_labeled_data.csv -> /content/drive/MyDrive/ECE324/youtube-thumbnail-performance-predictor-artifacts/merged_labeled_data.csv
Copied merged_ocr_features.csv -> /content/drive/MyDrive/ECE324/youtube-thumbnail-performance-predictor-artifacts/merged_ocr_features.csv
Copied merged_text_embeddings.npy -> /content/drive/MyDrive/ECE324/youtube-thumbnail-performance-predictor-artifacts/merged_text_embeddings.npy
Copied merged_face_embeddings.npy -> /content/drive/MyDrive/ECE324/youtube-thumbnail-performance-predictor-artifacts/merged_face_embeddings.npy


In [ ]:
print("No git pull in Colab after artifacts are generated. Use Drive sync instead.")

error: You have not concluded your merge (MERGE_HEAD exists).
hint: Please, commit your changes before merging.
fatal: Exiting because of unfinished merge.


In [ ]:
print("Skip git add/commit/push in Colab. Run the Drive export cell above instead.")

hint: You've added another git repository inside your current repository.
hint: Clones of the outer repository will not contain the contents of
hint: the embedded repository and will not know how to obtain it.
hint: If you meant to add a submodule, use:
hint: 
hint: 	git submodule add <url> youtube-thumbnail-performance-predictor
hint: 
hint: If you added this path by mistake, you can remove it from the
hint: index with:
hint: 
hint: 	git rm --cached youtube-thumbnail-performance-predictor
hint: 
hint: See "git help submodule" for more information.
[regression cc2e5c6] Final training run run.
 7 files changed, 13821 insertions(+)
 create mode 100644 data/processed/fusion_mlp_regression_metrics.json
 create mode 100644 data/processed/fusion_regression_tuning_all_results.csv
 create mode 100644 data/processed/fusion_regression_tuning_summary.csv
 create mode 100644 data/processed/fusion_regression_tuning_summary.json
 create mode 100644 data/processed/merged_cnn_cache_resnet50.csv
 creat

In [ ]:
print("No git cleanup/amend step needed in Colab when artifacts are stored on Drive.")

rm 'data/processed/merged_cnn_cache_resnet50.csv'
rm 'data/processed/fusion_mlp_regression_metrics.json'
rm 'data/processed/fusion_regression_tuning_all_results.csv'
rm 'data/processed/fusion_regression_tuning_summary.csv'
rm 'data/processed/fusion_regression_tuning_summary.json'
rm 'models/fusion_mlp_regression.pt'
[regression 60a7064] Final training run run.
 Date: Sun Mar 29 03:22:40 2026 +0000
 1 file changed, 1 insertion(+)
 create mode 160000 youtube-thumbnail-performance-predictor
Enumerating objects: 3, done.
Counting objects: 100% (3/3), done.
Delta compression using up to 12 threads
Compressing objects: 100% (2/2), done.
Writing objects: 100% (2/2), 300 bytes | 300.00 KiB/s, done.
Total 2 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/aria-yang/youtube-thumbnail-performance-predictor.git
   5ab832d..60a7064  regression -> regression


In [ ]:
print("No history rewrite needed here. Keep Colab for running jobs and Google Drive for storing outputs.")

/content/youtube-thumbnail-performance-predictor
	 rewrites.  Hit Ctrl-C before proceeding to abort, then use an
	 alternative filtering tool such as 'git filter-repo'
	 (https://github.com/newren/git-filter-repo/) instead.  See the
	 filter-branch manual page for more details; to squelch this warning,
	 set FILTER_BRANCH_SQUELCH_WARNING=1.
^C
Enumerating objects: 14393, done.
Counting objects: 100% (14393/14393), done.
Delta compression using up to 12 threads
^C
Everything up-to-date


In [ ]:
print("Artifact export is complete. If you need code changes in Git, commit and push them from your local machine.")

On branch regression
Your branch is up to date with 'origin/regression'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/processed/merged_cnn_cache_resnet50.csv

nothing added to commit but untracked files present (use "git add" to track)
